---
## 3. Preparação dos Dados

In [70]:
# Preparação dos dados
print("\n PREPARAÇÃO DOS DADOS")
print("="*60)

# Codificar labels
label_encoder = LabelEncoder()
df['label'] = label_encoder.fit_transform(df['class'])

print("\n Mapeamento de Classes:")
for i, cls in enumerate(label_encoder.classes_):
    print(f"   {i:2d} → {cls}")

# Divisão treino/teste estratificada (70/30 conforme trabalho de referência, com validação 10%)
X = df['path'].values
y = df['label'].values

# Primeiro split: 80% treino, 20% teste
X_train_paths, X_test_paths, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y
)

print(f"\n DIVISÃO DOS DADOS (conforme trabalho de referência):")
print(f"   Treino: {len(X_train_paths)} imagens ({len(X_train_paths)/len(df)*100:.1f}%)")
print(f"   Teste:  {len(X_test_paths)} imagens ({len(X_test_paths)/len(df)*100:.1f}%)")

# Verificar estratificação
print(f"\n Estratificação verificada:")
train_dist = pd.Series(y_train).value_counts().sort_index()
test_dist = pd.Series(y_test).value_counts().sort_index()
print(f"   Treino - Classes balanceadas: {train_dist.min()} a {train_dist.max()} por classe")
print(f"   Teste  - Classes balanceadas: {test_dist.min()} a {test_dist.max()} por classe")


 PREPARAÇÃO DOS DADOS

 Mapeamento de Classes:
    0 → Agglutinated
    1 → Brittle
    2 → Compartmentalized_Brown
    3 → Compartmentalized_PartiallyPurple
    4 → Compartmentalized_Purple
    5 → Compartmentalized_Slaty
    6 → Compartmentalized_White
    7 → Flattened
    8 → Moldered
    9 → Plated_Brown
   10 → Plated_PartiallyPurple
   11 → Plated_Purple
   12 → Plated_Slaty
   13 → Plated_White

 DIVISÃO DOS DADOS (conforme trabalho de referência):
   Treino: 1120 imagens (80.0%)
   Teste:  280 imagens (20.0%)

 Estratificação verificada:
   Treino - Classes balanceadas: 80 a 80 por classe
   Teste  - Classes balanceadas: 20 a 20 por classe


---
## 4. Extração de Características Visuais

Conforme solicitado no trabalho pedagógico, foram utilizadas as arquiteturas mencionadas no trabalho correlato (Malcher & Guedes, 2022) **sem as camadas finais de classificação** e com **pesos fixos**, para produzir vetores de características visuais.

### Arquiteturas do Trabalho de Referência

O trabalho correlato utilizou: LeNet, AlexNet, MobileNet, VGG-16, Inception, EfficientNetB0 e EfficientNetB3.

### Arquiteturas Implementadas

| CNN | Dim. Features | Input Size | Observação |
|-----|---------------|------------|------------|
| InceptionV3 | 2048 | 299x299 | Top 3 do trabalho de referência |
| VGG16 | 512 | 224x224 | Top 3 do trabalho de referência |
| EfficientNetB0 | 1280 | 224x224 | Top 3 do trabalho de referência |
| MobileNetV2 | 1280 | 224x224 | MobileNet do trabalho de referência |
| EfficientNetB3 | 1536 | 300x300 | Conforme trabalho de referência |
| DenseNet121 | 1024 | 224x224 | Substitui LeNet/AlexNet (não disponíveis no Keras) |

**Nota:** LeNet e AlexNet não estão disponíveis no Keras Applications como modelos pré-treinados no ImageNet. Como o objetivo é usar CNNs como extratoras de características com pesos pré-treinados, incluímos DenseNet121 como alternativa.

In [71]:
# Definição das arquiteturas CNN - TODAS as arquiteturas do trabalho de referência (Malcher & Guedes, 2022)
# Arquiteturas listadas no trabalho correlato: LeNet, AlexNet, MobileNet, VGG-16, Inception, EfficientNetB0, EfficientNetB3
# Nota: LeNet e AlexNet não estão disponíveis no Keras Applications, então usamos implementações equivalentes

cnn_models = {
    # CNNs Top 3 do trabalho de referência (melhores resultados)
    'InceptionV3': {
        'model_fn': InceptionV3,
        'preprocess': inception_preprocess,
        'input_size': (299, 299),
        'feature_dim': 2048
    },
    'VGG16': {
        'model_fn': VGG16,
        'preprocess': vgg_preprocess,
        'input_size': (224, 224),
        'feature_dim': 512
    },
    'EfficientNetB0': {
        'model_fn': EfficientNetB0,
        'preprocess': efficientnet_preprocess,
        'input_size': (224, 224),
        'feature_dim': 1280
    },
    # Demais CNNs do trabalho de referência
    'MobileNetV2': {  # MobileNet mencionada no trabalho
        'model_fn': MobileNetV2,
        'preprocess': mobilenet_preprocess,
        'input_size': (224, 224),
        'feature_dim': 1280
    },
    'EfficientNetB3': {
        'model_fn': EfficientNetB3,
        'preprocess': efficientnet_preprocess,
        'input_size': (300, 300),
        'feature_dim': 1536
    },
    # LeNet e AlexNet não disponíveis no Keras - substituídas por arquiteturas similares
    # DenseNet121 como alternativa para extração de features
    'DenseNet121': {
        'model_fn': DenseNet121,
        'preprocess': densenet_preprocess,
        'input_size': (224, 224),
        'feature_dim': 1024
    }
}

print("\n ARQUITETURAS CNN CONFIGURADAS:")
print("="*70)
print(f"{'CNN':<15} {'Input Size':<12} {'Features':<10} {'Observação'}")
print("-"*70)
for name, config in cnn_models.items():
    nota = "(Top 3)" if name in ['InceptionV3', 'VGG16', 'EfficientNetB0'] else ""
    print(f"{name:<15} {str(config['input_size']):<12} {config['feature_dim']:<10} {nota}")
print("="*70)
print(f"\nTotal de arquiteturas: {len(cnn_models)}")
print("\nNota: LeNet e AlexNet do trabalho de referência não estão disponíveis")
print("no Keras Applications. DenseNet121 foi incluída como alternativa.")


 ARQUITETURAS CNN CONFIGURADAS:
CNN             Input Size   Features   Observação
----------------------------------------------------------------------
InceptionV3     (299, 299)   2048       (Top 3)
VGG16           (224, 224)   512        (Top 3)
EfficientNetB0  (224, 224)   1280       (Top 3)
MobileNetV2     (224, 224)   1280       
EfficientNetB3  (300, 300)   1536       
DenseNet121     (224, 224)   1024       

Total de arquiteturas: 6

Nota: LeNet e AlexNet do trabalho de referência não estão disponíveis
no Keras Applications. DenseNet121 foi incluída como alternativa.


In [73]:
# Função para extrair características - VERSÃO OTIMIZADA COM BATCH
def extract_features(image_paths, cnn_name, batch_size=16, verbose=True):
    """
    Extrai características visuais usando uma CNN pré-treinada.
    Versão otimizada com processamento em batch.
    """
    config = cnn_models[cnn_name]
    input_size = config['input_size']
    preprocess = config['preprocess']

    # Carregar modelo base (sem camadas de classificação)
    base_model = config['model_fn'](weights='imagenet', include_top=False, pooling='avg')

    if verbose:
        print(f"\n   Extraindo características com {cnn_name}...")
        print(f"   Input size: {input_size} | Batch size: {batch_size}")

    features = []
    n_images = len(image_paths)
    n_batches = (n_images + batch_size - 1) // batch_size

    for batch_idx in range(n_batches):
        start_idx = batch_idx * batch_size
        end_idx = min(start_idx + batch_size, n_images)
        batch_paths = image_paths[start_idx:end_idx]

        # Carregar e preprocessar batch de imagens
        batch_images = []
        for img_path in batch_paths:
            img = image.load_img(img_path, target_size=input_size)
            img_array = image.img_to_array(img)
            batch_images.append(img_array)

        batch_array = np.array(batch_images)
        batch_array = preprocess(batch_array)

        # Extrair features do batch inteiro de uma vez
        batch_features = base_model.predict(batch_array, verbose=0)
        features.extend(batch_features)

        if verbose and (batch_idx + 1) % 10 == 0:
            print(f"   Processados {end_idx}/{n_images} imagens ({end_idx/n_images*100:.1f}%)")

    features = np.array(features)
    if verbose:
        print(f"   Concluido! Shape: {features.shape}")

    return features

---
## 5. Feature Fusion (Concatenação de Features)

Concatenação das características extraídas das 3 melhores CNNs (InceptionV3, VGG16, EfficientNetB0) para formar um vetor de features combinado.

In [93]:
# Feature Fusion - Concatenar features das CNNs do trabalho de referência
print("\n" + "="*70)
print(" FEATURE FUSION - CNNs DO TRABALHO DE REFERÊNCIA")
print("="*70)

# CNNs usadas no trabalho de referência (Top 3)
reference_cnns = ['InceptionV3', 'VGG16', 'EfficientNetB0']

print(f"\nConcatenando features de: {reference_cnns}")

# Concatenar features
X_train_fused = np.hstack([features_dict[cnn]['X_train'] for cnn in reference_cnns])
X_test_fused = np.hstack([features_dict[cnn]['X_test'] for cnn in reference_cnns])

print(f"\n Dimensões após fusão:")
print(f"   Treino: {X_train_fused.shape}")
print(f"   Teste:  {X_test_fused.shape}")
print(f"   Total de features: {X_train_fused.shape[1]} (2048 + 512 + 1280)")

# Normalização
print(f"\n Aplicando normalização (StandardScaler)...")
scaler_fused = StandardScaler()
X_train_fused_scaled = scaler_fused.fit_transform(X_train_fused)
X_test_fused_scaled = scaler_fused.transform(X_test_fused)
print("    Dados normalizados")


 FEATURE FUSION - CNNs DO TRABALHO DE REFERÊNCIA

Concatenando features de: ['InceptionV3', 'VGG16', 'EfficientNetB0']

 Dimensões após fusão:
   Treino: (1120, 3840)
   Teste:  (280, 3840)
   Total de features: 3840 (2048 + 512 + 1280)

 Aplicando normalização (StandardScaler)...
    Dados normalizados


---
## 6. Redução de Dimensionalidade com PCA

A Análise de Componentes Principais (PCA) foi aplicada para:

1. Remover ruído presente nas features de alta dimensionalidade
2. Eliminar redundância entre características de diferentes CNNs
3. Reduzir o custo computacional do treinamento das SVMs
4. Melhorar a generalização ao focar nas componentes mais informativas

In [75]:
# PCA para reduzir dimensionalidade da fusão
print("\n" + "="*70)
print(" REDUÇÃO DE DIMENSIONALIDADE COM PCA")
print("="*70)

# Testar diferentes números de componentes
pca_results = []
n_samples = X_train_fused_scaled.shape[0]

print(f"\nNúmero de amostras de treino: {n_samples}")
print(f"Número de features original: {X_train_fused_scaled.shape[1]}")
print(f"\nTestando diferentes valores de n_components...\n")

for n_components in [50, 100, 200, 300, 700]:
    if n_components > min(n_samples, X_train_fused_scaled.shape[1]):
        continue

    pca = PCA(n_components=n_components, random_state=RANDOM_STATE)
    X_train_pca = pca.fit_transform(X_train_fused_scaled)
    X_test_pca = pca.transform(X_test_fused_scaled)

    variance_explained = pca.explained_variance_ratio_.sum()

    # Teste rápido com SVM RBF
    svm_quick = SVC(kernel='rbf', C=10, gamma='scale', random_state=RANDOM_STATE)
    svm_quick.fit(X_train_pca, y_train)
    acc = accuracy_score(y_test, svm_quick.predict(X_test_pca))

    pca_results.append({
        'n_components': n_components,
        'variance': variance_explained,
        'accuracy': acc
    })

    print(f"   PCA({n_components:3d}): variância={variance_explained:.2%}, acc={acc:.2%}")

# Escolher melhor PCA
best_pca_config = max(pca_results, key=lambda x: x['accuracy'])
print(f"\n Melhor PCA: {best_pca_config['n_components']} componentes")
print(f"   Variância explicada: {best_pca_config['variance']:.2%}")
print(f"   Acurácia: {best_pca_config['accuracy']:.2%}")


 REDUÇÃO DE DIMENSIONALIDADE COM PCA

Número de amostras de treino: 1120
Número de features original: 3840

Testando diferentes valores de n_components...

   PCA( 50): variância=55.07%, acc=72.14%
   PCA(100): variância=65.89%, acc=72.86%
   PCA(200): variância=77.12%, acc=75.36%
   PCA(300): variância=83.93%, acc=76.07%
   PCA(700): variância=96.35%, acc=77.86%

 Melhor PCA: 700 componentes
   Variância explicada: 96.35%
   Acurácia: 77.86%


In [ ]:
# Teste sem PCA
svm_no_pca = SVC(kernel='rbf', C=10, gamma='scale', random_state=RANDOM_STATE)
svm_no_pca.fit(X_train_fused_scaled, y_train)  # features originais, sem PCA
acc_no_pca = accuracy_score(y_test, svm_no_pca.predict(X_test_fused_scaled))
print(f"Sem PCA (3840 features): {acc_no_pca:.2%}")

Sem PCA (3840 features): 75.36%


In [76]:
# Aplicar PCA com melhor configuração
best_n_components = best_pca_config['n_components']

print(f"\n Aplicando PCA com {best_n_components} componentes...")
pca_final = PCA(n_components=best_n_components, random_state=RANDOM_STATE)
X_train_pca_final = pca_final.fit_transform(X_train_fused_scaled)
X_test_pca_final = pca_final.transform(X_test_fused_scaled)

print(f"   Shape treino: {X_train_pca_final.shape}")
print(f"   Shape teste:  {X_test_pca_final.shape}")
print(f"   Variância explicada: {pca_final.explained_variance_ratio_.sum():.2%}")


 Aplicando PCA com 700 componentes...
   Shape treino: (1120, 700)
   Shape teste:  (280, 700)
   Variância explicada: 96.35%
